# MetalloDock 虚拟筛选教程

**论文引用:** MetalloDock -- 面向金属蛋白的分子对接方法，专门针对含金属离子的蛋白结合位点设计。

## 核心创新

MetalloDock 的核心思想是针对 **金属蛋白 (metalloprotein)** 的对接场景：

1. 利用 **BFS（广度优先搜索）** 确定配体原子的生成顺序；
2. 以 **自回归 (autoregressive)** 方式逐原子预测配体构象——每一步已知父原子坐标和键长，MLP 预测位移方向；
3. 新坐标 = 父原子坐标 + 归一化位移方向 x 键长；
4. 同时预测哪个配体原子最可能是 **金属配位原子 (coordination donor)**。

## 学习目标

通过本教程，你将掌握：

- **自回归生成**：原子按 BFS 拓扑顺序依次放置，每步利用已放置原子的上下文
- **Teacher Forcing**：训练时使用真实坐标计算上下文，推理时用预测坐标
- **配位原子预测**：辅助任务，帮助模型理解金属蛋白的结合模式
- **蛋白-配体上下文交互层**：距离加权注意力机制实现蛋白到配体信息传递
- **RMSD 评估**：对接任务的标准评估指标

In [ ]:
# ============================================================
# 项目根目录定位、路径设置与依赖导入
# ============================================================
import os
import warnings
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit import RDLogger
from scipy.spatial import distance_matrix

%matplotlib inline
import matplotlib.pyplot as plt

# 抑制 RDKit 的冗余警告信息
RDLogger.logger().setLevel(RDLogger.ERROR)
warnings.filterwarnings('ignore')


def find_project_root(marker='demo_data'):
    """向上查找包含 demo_data/ 的项目根目录"""
    here = Path('.').resolve()
    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f'无法找到包含 {marker}/ 的项目根目录')


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / 'demo_data'
CORESET_FILE = DATA_DIR / 'CoreSet.dat'
COMPLEX_DIR = DATA_DIR / 'coreset'

print(f'项目根目录: {PROJECT_ROOT}')
print(f'数据目录:   {DATA_DIR}')

# 版本信息
version_info = pd.DataFrame({
    '库': ['torch', 'rdkit', 'numpy', 'scipy', 'pandas'],
    '版本': [
        torch.__version__,
        Chem.rdBase.rdkitVersion,
        np.__version__,
        __import__('scipy').__version__,
        pd.__version__,
    ]
})
display(version_info)

## 1. 超参数设置

MetalloDock 的关键超参数包括：

| 参数 | 说明 |
|------|------|
| `ATOM_FEAT_DIM` | 简化原子特征维度（9 种元素 one-hot + 芳香性） |
| `HIDDEN_DIM` | 隐层维度 |
| `POCKET_CUTOFF` | 截取口袋原子的距离阈值 |
| `INTERACTION_CUTOFF` | 蛋白-配体交互距离阈值 |
| `N_CONTEXT_LAYERS` | 上下文交互层数 |
| `COORD_NOISE_STD` | 训练时对配体坐标添加的高斯噪声标准差（增强鲁棒性） |

In [ ]:
# ============================================================
# 超参数定义
# ============================================================
ATOM_FEAT_DIM = 10          # 简化原子特征维度
HIDDEN_DIM = 128            # 隐层维度
N_EPOCHS = 200              # 训练轮数
LR = 1e-3                   # 学习率
BATCH_SIZE = 1              # 变长结构，逐样本处理
SEED = 42
POCKET_CUTOFF = 8.0         # 截取口袋原子的距离阈值 (Angstrom)
INTERACTION_CUTOFF = 6.0    # 蛋白-配体交互距离阈值 (Angstrom)
N_CONTEXT_LAYERS = 4        # 上下文交互层数
COORD_NOISE_STD = 0.3       # 训练时对配体坐标添加的噪声标准差
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)

# 展示超参数表格
hp_df = pd.DataFrame({
    '超参数': [
        'ATOM_FEAT_DIM', 'HIDDEN_DIM', 'N_EPOCHS', 'LR', 'BATCH_SIZE',
        'POCKET_CUTOFF', 'INTERACTION_CUTOFF', 'N_CONTEXT_LAYERS',
        'COORD_NOISE_STD', 'DEVICE'
    ],
    '值': [
        ATOM_FEAT_DIM, HIDDEN_DIM, N_EPOCHS, LR, BATCH_SIZE,
        f'{POCKET_CUTOFF} A', f'{INTERACTION_CUTOFF} A', N_CONTEXT_LAYERS,
        COORD_NOISE_STD, str(DEVICE)
    ],
    '说明': [
        '简化原子特征维度', '隐层维度', '训练轮数', '学习率', '变长结构，逐样本处理',
        '截取口袋原子的距离阈值', '蛋白-配体交互距离阈值', '上下文交互层数',
        '训练时对配体坐标添加的噪声标准差', '计算设备'
    ]
})
display(hp_df)

## 2. 数据加载与特征提取

MetalloDock 的数据处理流程：

1. **解析 CoreSet.dat** 获取有效 PDB ID 列表（MetalloDock 是对接模型，logKa 仅用于筛选复合物）
2. **加载蛋白口袋与配体** 的 3D 结构
3. **提取原子特征**：9 类元素 one-hot + 芳香性 = 10 维
4. **BFS 排序**：确定原子的自回归生成顺序
5. **提取键长**：作为生成时的距离约束
6. **配位原子标签**：离蛋白最近的 N/O/S 配体原子标为 1（简化版）

In [ ]:
# ============================================================
# 特征提取函数
# ============================================================

# ---- 元素符号 -> one-hot 映射 (9类 + 1 芳香性 = 10维) ----
ELEMENT_LIST = ['C', 'N', 'O', 'S', 'F', 'P', 'Cl', 'Br']  # 8 种常见元素 + 1 other


def parse_coreset(path):
    """解析 CoreSet.dat，返回 {pdbid: logKa} 字典。跳过以 '#' 开头的注释行。
    注意：MetalloDock 是对接模型，logKa 仅用于获取有效 PDB ID 列表。"""
    labels = {}
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            pdbid = parts[0]
            logKa = float(parts[3])
            labels[pdbid] = logKa
    print(f"[INFO] 从 CoreSet.dat 读取到 {len(labels)} 个复合物")
    return labels


def atom_features(atom):
    """
    构建 10 维原子特征向量：
      - 前 9 维: 元素类型 one-hot (C, N, O, S, F, P, Cl, Br, other)
      - 第 10 维: 是否为芳香性原子
    """
    feat = np.zeros(ATOM_FEAT_DIM, dtype=np.float32)
    symbol = atom.GetSymbol()
    if symbol in ELEMENT_LIST:
        feat[ELEMENT_LIST.index(symbol)] = 1.0
    else:
        feat[8] = 1.0       # other 类别
    feat[9] = float(atom.GetIsAromatic())
    return feat


def load_mol(path, fmt):
    """
    用 RDKit 加载分子文件。
    fmt: 'mol2', 'sdf', 'pdb'
    先尝试正常加载，再尝试 sanitize=False 后手动 sanitize。
    """
    mol = None
    if fmt == 'mol2':
        mol = Chem.MolFromMol2File(path, sanitize=True)
        if mol is None:
            mol = Chem.MolFromMol2File(path, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass  # 保留 unsanitized 的分子
    elif fmt == 'sdf':
        supplier = Chem.SDMolSupplier(path, sanitize=True)
        for m in supplier:
            if m is not None:
                mol = m
                break
        if mol is None:
            supplier = Chem.SDMolSupplier(path, sanitize=False)
            for m in supplier:
                if m is not None:
                    mol = m
                    try:
                        Chem.SanitizeMol(mol)
                    except Exception:
                        pass
                    break
    elif fmt == 'pdb':
        mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=True)
        if mol is None:
            mol = Chem.MolFromPDBFile(path, removeHs=False, sanitize=False)
            if mol is not None:
                try:
                    Chem.SanitizeMol(mol)
                except Exception:
                    pass
    return mol


def compute_rmsd(pred_coords, true_coords):
    """计算两组坐标之间的 RMSD (均方根偏差)。
    pred_coords, true_coords: (N, 3) numpy 数组
    公式: RMSD = sqrt( mean( sum_i (pred_i - true_i)^2 ) )
    """
    diff = pred_coords - true_coords
    return np.sqrt(np.mean(np.sum(diff ** 2, axis=1)))


def bfs_atom_order(mol, start_idx=0):
    """广度优先搜索确定原子生成顺序（参考 MetalloDock 的 BFS_search）。

    核心思想：从起始原子出发，沿化学键拓扑顺序 BFS 遍历整个分子。
    返回的 atom_order 就是自回归生成时放置原子的顺序，
    parent_list 记录每个原子的"父原子"——即从哪个已放置的原子延伸而来。

    参数:
      mol       : RDKit Mol 对象
      start_idx : BFS 起始原子索引

    返回:
      atom_order  : list[int] -- 原子生成顺序
      parent_list : list[int] -- 每个原子的父原子索引（第一个原子的父为 -1）
    """
    num_atoms = mol.GetNumAtoms()
    visited = [False] * num_atoms
    atom_order = [start_idx]
    parent_list = [-1]  # 第一个原子没有父原子
    visited[start_idx] = True

    queue = deque([start_idx])
    while queue:
        current = queue.popleft()
        for neighbor in mol.GetAtomWithIdx(current).GetNeighbors():
            nb_idx = neighbor.GetIdx()
            if not visited[nb_idx]:
                visited[nb_idx] = True
                queue.append(nb_idx)
                atom_order.append(nb_idx)
                parent_list.append(current)

    return atom_order, parent_list


def get_bond_lengths(mol, conf):
    """提取分子中所有键的键长，返回 {(i,j): length} 字典。
    用于自回归生成时确定新原子与父原子之间的距离约束。"""
    bond_lengths = {}
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        pi = np.array(conf.GetAtomPosition(i))
        pj = np.array(conf.GetAtomPosition(j))
        length = float(np.linalg.norm(pi - pj))
        bond_lengths[(i, j)] = length
        bond_lengths[(j, i)] = length
    return bond_lengths


def build_metallodock_data(pdbid):
    """
    为单个蛋白-配体复合物构建 MetalloDock 数据。

    返回:
      prot_feats   : (N_p, 10)   蛋白口袋原子特征
      lig_feats    : (N_l, 10)   配体原子特征
      prot_coords  : (N_p, 3)    蛋白口袋原子坐标
      lig_coords   : (N_l, 3)    配体原子坐标 (晶体结构真实坐标)
      atom_order   : list[int]   BFS 原子生成顺序
      parent_list  : list[int]   每个原子的父原子索引
      bond_lengths : dict        键长字典 {(i,j): float}
      donor_label  : (N_l,)      配位原子标签 (简化版: 离蛋白最近的 N/O/S 原子标为 1)
    如果解析失败则返回 None。
    """
    pocket_path = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_pocket.pdb")
    ligand_mol2 = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.mol2")
    ligand_sdf = os.path.join(COMPLEX_DIR, pdbid, f"{pdbid}_ligand.sdf")

    # ---- 1. 加载蛋白口袋 ----
    prot_mol = Chem.MolFromPDBFile(pocket_path, removeHs=True, sanitize=False)
    if prot_mol is None:
        return None
    try:
        Chem.SanitizeMol(prot_mol)
    except Exception:
        pass

    # ---- 2. 加载配体 (mol2 优先, sdf 备选) ----
    lig_mol = Chem.MolFromMol2File(ligand_mol2, sanitize=False)
    if lig_mol is None and os.path.exists(ligand_sdf):
        lig_mol = load_mol(ligand_sdf, 'sdf')
    if lig_mol is None:
        return None
    try:
        Chem.SanitizeMol(lig_mol)
    except Exception:
        pass

    # ---- 3. 去氢 ----
    try:
        prot_mol = Chem.RemoveHs(prot_mol)
    except Exception:
        pass
    try:
        lig_mol = Chem.RemoveHs(lig_mol)
    except Exception:
        pass

    if lig_mol.GetNumAtoms() < 2:
        return None  # 至少需要 2 个原子才能做 BFS

    # ---- 4. 提取 3D 坐标与原子特征 ----
    prot_conf = prot_mol.GetConformer()
    lig_conf = lig_mol.GetConformer()

    prot_coords = np.array([prot_conf.GetAtomPosition(i)
                            for i in range(prot_mol.GetNumAtoms())], dtype=np.float32)
    lig_coords = np.array([lig_conf.GetAtomPosition(i)
                           for i in range(lig_mol.GetNumAtoms())], dtype=np.float32)

    prot_feats = np.array([atom_features(a) for a in prot_mol.GetAtoms()], dtype=np.float32)
    lig_feats = np.array([atom_features(a) for a in lig_mol.GetAtoms()], dtype=np.float32)

    if prot_feats.shape[0] == 0 or lig_feats.shape[0] == 0:
        return None

    # ---- 5. 截取离配体较近的口袋原子 ----
    lig_center = lig_coords.mean(axis=0)
    prot_dists_to_center = np.linalg.norm(prot_coords - lig_center, axis=1)
    pocket_mask = prot_dists_to_center < POCKET_CUTOFF
    if pocket_mask.sum() < 3:
        top_k = min(20, len(prot_coords))
        pocket_idx = np.argsort(prot_dists_to_center)[:top_k]
        prot_coords = prot_coords[pocket_idx]
        prot_feats = prot_feats[pocket_idx]
    else:
        prot_coords = prot_coords[pocket_mask]
        prot_feats = prot_feats[pocket_mask]

    # ---- 6. BFS 原子排序 ----
    # 在完整 MetalloDock 中，起始原子是预测的配位原子（离金属最近的配体原子）
    # 简化版：选择离蛋白口袋中心最近的配体原子作为起始
    dists_to_pocket = np.linalg.norm(lig_coords - prot_coords.mean(axis=0), axis=1)
    start_idx = int(np.argmin(dists_to_pocket))
    atom_order, parent_list = bfs_atom_order(lig_mol, start_idx=start_idx)

    # ---- 7. 提取键长 ----
    bond_lengths = get_bond_lengths(lig_mol, lig_conf)

    # ---- 8. 配位原子标签 (简化版) ----
    # 真实 MetalloDock 使用金属-配体距离定义配位原子
    # 简化：离蛋白最近的 N/O/S 配体原子标为 1（模拟配位倾向）
    donor_label = np.zeros(lig_mol.GetNumAtoms(), dtype=np.float32)
    donor_elements = {'N', 'O', 'S'}
    min_dist_to_prot = distance_matrix(lig_coords, prot_coords).min(axis=1)
    for i, atom in enumerate(lig_mol.GetAtoms()):
        if atom.GetSymbol() in donor_elements and min_dist_to_prot[i] < 3.5:
            donor_label[i] = 1.0

    return (prot_feats, lig_feats, prot_coords, lig_coords,
            atom_order, parent_list, bond_lengths, donor_label)

In [ ]:
# ============================================================
# 加载数据并展示一个样本
# ============================================================
print("正在构建 MetalloDock 数据...")
labels = parse_coreset(CORESET_FILE)

all_data = []
failed = 0
for pdbid in sorted(labels.keys()):
    result = build_metallodock_data(pdbid)
    if result is None:
        failed += 1
        continue
    all_data.append(result)

print(f"成功加载 {len(all_data)} 个复合物, 失败 {failed} 个")

# 展示第一个样本的基本信息
sample = all_data[0]
sample_info = pd.DataFrame({
    '属性': [
        '蛋白口袋原子数', '配体原子数',
        '蛋白特征维度', '配体特征维度',
        'BFS 序列长度', '键数量',
        '配位原子数'
    ],
    '值': [
        sample[0].shape[0], sample[1].shape[0],
        sample[0].shape[1], sample[1].shape[1],
        len(sample[4]), len(sample[6]) // 2,
        int(sample[7].sum())
    ]
})
display(sample_info)

## 3. 数据集与数据加载器

由于每个蛋白-配体复合物的原子数不同（变长结构），我们使用 `BATCH_SIZE=1` 逐样本处理。

数据集的 `__getitem__` 方法将 BFS 序列中每对 `(child, parent)` 的键长提取为数组 `bond_len_seq`，供自回归生成时使用。

In [ ]:
# ============================================================
# Dataset 与 DataLoader
# ============================================================

class MetalloDockDataset(Dataset):
    """MetalloDock 数据集。每个样本包含蛋白/配体数据及 BFS 生成序列。
    这是对接任务，训练目标是逐原子坐标预测。"""
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        (prot_f, lig_f, prot_c, lig_c,
         atom_order, parent_list, bond_lengths, donor_label) = self.data[idx]

        n_lig = len(atom_order)

        # 将 BFS 序列中每对 (child, parent) 的键长提取为数组
        # bond_len_seq[i] 是第 i 步生成的原子与其父原子之间的键长
        bond_len_seq = np.zeros(n_lig, dtype=np.float32)
        for i in range(1, n_lig):
            child = atom_order[i]
            parent = parent_list[i]
            bl = bond_lengths.get((child, parent), 1.5)  # 默认键长 1.5A
            bond_len_seq[i] = bl

        return (torch.FloatTensor(prot_f),          # (N_p, 10)
                torch.FloatTensor(lig_f),           # (N_l, 10)
                torch.FloatTensor(prot_c),          # (N_p, 3)
                torch.FloatTensor(lig_c),           # (N_l, 3)
                torch.LongTensor(atom_order),       # (N_l,)
                torch.LongTensor(parent_list),      # (N_l,)
                torch.FloatTensor(bond_len_seq),    # (N_l,)
                torch.FloatTensor(donor_label))     # (N_l,)


# ---- 随机 80/20 划分训练/测试集 ----
indices = np.random.permutation(len(all_data))
split = int(0.8 * len(all_data))
train_idx, test_idx = indices[:split], indices[split:]

train_data = [all_data[i] for i in train_idx]
test_data = [all_data[i] for i in test_idx]

train_loader = DataLoader(MetalloDockDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(MetalloDockDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# 展示数据集划分信息
split_df = pd.DataFrame({
    '数据集': ['训练集', '测试集', '总计'],
    '样本数': [len(train_data), len(test_data), len(all_data)]
})
display(split_df)

# 展示一个 batch 的张量形状
sample_batch = next(iter(train_loader))
shape_df = pd.DataFrame({
    '张量': [
        'prot_feats', 'lig_feats', 'prot_coords', 'lig_coords',
        'atom_order', 'parent_list', 'bond_len_seq', 'donor_label'
    ],
    '形状': [str(tuple(t.shape)) for t in sample_batch],
    '说明': [
        '蛋白口袋原子特征', '配体原子特征',
        '蛋白口袋原子坐标', '配体原子坐标（真实）',
        'BFS 原子生成顺序', '每个原子的父原子索引',
        '每步的键长', '配位原子标签'
    ]
})
display(shape_df)

## 4. 模型架构

ToyMetalloDock 模型的核心组件：

### 4.1 ContextInteractionLayer（上下文交互层）

在自回归生成的每一步，已放置的原子需要与蛋白口袋"交流"以感知周围环境。该层通过距离加权的注意力实现蛋白到配体信息传递：
1. 计算已放置配体原子与蛋白原子的距离
2. 距离越近的蛋白原子贡献越大（注意力权重）
3. 更新配体原子的隐表示

### 4.2 ToyMetalloDock（主模型）

自回归逐原子坐标预测的核心流程：
1. BFS 确定原子生成顺序（从配位原子出发）
2. 对每个待生成原子，已知其父原子坐标和键长
3. 用多层上下文交互层编码已放置原子与蛋白的上下文
4. MLP 预测当前原子相对于父原子的 3D 位移方向
5. 新坐标 = 父原子坐标 + 归一化(位移方向) * 键长

**辅助任务**：`coord_donor_head` 预测每个配体原子是否是配位原子（BCE 损失），帮助模型关注金属配位相关的化学信息。

In [ ]:
# ============================================================
# 模型架构 -- ToyMetalloDock
# ============================================================

class ContextInteractionLayer(nn.Module):
    """蛋白-配体上下文交互层（简化版）。

    在自回归生成的每一步，已放置的原子需要与蛋白口袋"交流"，
    以感知周围环境。该层通过距离加权的注意力实现蛋白到配体信息传递：
      1. 计算已放置配体原子与蛋白原子的距离
      2. 距离越近的蛋白原子贡献越大（注意力权重）
      3. 更新配体原子的隐表示
    """
    def __init__(self, hidden_dim):
        super().__init__()
        # 消息 MLP: [h_lig || h_prot || dist] -> message
        self.msg_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        # 注意力权重
        self.attn_fc = nn.Linear(hidden_dim, 1)
        # 更新 MLP
        self.update_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, lig_h, prot_h, lig_coords, prot_coords, mask):
        """
        参数:
          lig_h       : (N_l, hidden)  配体原子隐表示
          prot_h      : (N_p, hidden)  蛋白原子隐表示
          lig_coords  : (N_l, 3)       已放置的配体原子坐标
          prot_coords : (N_p, 3)       蛋白原子坐标
          mask        : (N_l,)         bool 掩码，True 表示该原子已放置

        返回:
          lig_h_new   : (N_l, hidden)  更新后的配体隐表示
        """
        n_lig = lig_h.size(0)
        n_prot = prot_h.size(0)

        # 计算已放置配体原子与所有蛋白原子的距离
        # (N_l, N_p)
        dist = torch.cdist(lig_coords.unsqueeze(0), prot_coords.unsqueeze(0)).squeeze(0)

        # 对每个已放置的配体原子，聚合来自蛋白的消息
        # 扩展用于拼接: lig_h_exp (N_l, N_p, hidden), prot_h_exp (N_l, N_p, hidden)
        lig_h_exp = lig_h.unsqueeze(1).expand(-1, n_prot, -1)
        prot_h_exp = prot_h.unsqueeze(0).expand(n_lig, -1, -1)
        dist_feat = dist.unsqueeze(-1)  # (N_l, N_p, 1)

        msg_input = torch.cat([lig_h_exp, prot_h_exp, dist_feat], dim=-1)
        msg = self.msg_mlp(msg_input)  # (N_l, N_p, hidden)

        # 注意力加权（距离越近权重越高）
        attn = torch.softmax(self.attn_fc(msg).squeeze(-1), dim=-1)  # (N_l, N_p)
        agg = torch.bmm(attn.unsqueeze(1), msg).squeeze(1)  # (N_l, hidden)

        # 仅更新已放置的原子
        update = self.update_mlp(torch.cat([lig_h, agg], dim=-1))
        lig_h_new = self.norm(lig_h + update * mask.unsqueeze(-1).float())
        return lig_h_new


class ToyMetalloDock(nn.Module):
    """MetalloDock 玩具模型——自回归逐原子坐标预测。

    核心思想（参考 MetalloDock 原文的 sampling_atom_pos）：
      1. BFS 确定原子生成顺序（从配位原子出发）
      2. 对每个待生成原子，已知其父原子坐标和键长
      3. 用 EGNN 风格的交互层编码已放置原子与蛋白的上下文
      4. MLP 预测当前原子相对于父原子的 3D 位移方向
      5. 新坐标 = 父原子坐标 + 归一化(位移方向) * 键长

    辅助任务：
      coord_donor_head 预测每个配体原子是否是配位原子（BCE 损失），
      帮助模型关注金属配位相关的化学信息。
    """
    def __init__(self, atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM,
                 n_context_layers=N_CONTEXT_LAYERS):
        super().__init__()
        # 原子嵌入
        self.prot_embed = nn.Linear(atom_dim, hidden_dim)
        self.lig_embed = nn.Linear(atom_dim, hidden_dim)

        # 上下文交互层：编码已放置原子与蛋白的空间关系
        self.context_layers = nn.ModuleList([
            ContextInteractionLayer(hidden_dim)
            for _ in range(n_context_layers)
        ])

        # 位移方向预测头
        # 输入: [当前原子隐表示 || 父原子隐表示 || 键长]
        self.displacement_head = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, 3),
        )

        # 配位原子预测头
        self.coord_donor_head = nn.Sequential(
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid(),
        )

    def encode_context(self, lig_h, prot_h, placed_coords, prot_coords, mask):
        """多层上下文交互，编码已放置原子与蛋白的空间关系。"""
        for layer in self.context_layers:
            lig_h = layer(lig_h, prot_h, placed_coords, prot_coords, mask)
        return lig_h

    def predict_displacement(self, lig_h, child_idx, parent_idx, bond_len):
        """预测子原子相对于父原子的位移方向。

        参数:
          lig_h      : (N_l, hidden)  上下文编码后的配体隐表示
          child_idx  : int            待生成原子索引
          parent_idx : int            父原子索引
          bond_len   : float          键长

        返回:
          displacement : (3,)  位移向量（已归一化并乘以键长）
        """
        child_h = lig_h[child_idx]   # (hidden,)
        parent_h = lig_h[parent_idx]  # (hidden,)
        bl_feat = torch.tensor([bond_len], device=lig_h.device, dtype=lig_h.dtype)

        inp = torch.cat([child_h, parent_h, bl_feat], dim=-1)  # (hidden*2+1,)
        raw_disp = self.displacement_head(inp)  # (3,)

        # 归一化位移方向，乘以键长
        direction = raw_disp / (raw_disp.norm() + 1e-8)
        return direction * bond_len

    def forward_train(self, prot_feats, lig_feats, prot_coords, lig_coords_true,
                      atom_order, parent_list, bond_len_seq):
        """训练前向传播（Teacher Forcing）。

        使用真实坐标作为已放置原子的坐标，逐步预测每个原子的位移。
        这避免了自回归中误差累积的问题，加速训练收敛。

        返回:
          per_atom_loss : (N_l-1,)  每个原子的坐标预测 L2 误差
          donor_pred    : (N_l,)    配位原子预测概率
        """
        n_lig = lig_feats.size(0)

        # 嵌入
        prot_h = self.prot_embed(prot_feats)  # (N_p, hidden)
        lig_h = self.lig_embed(lig_feats)     # (N_l, hidden)

        # Teacher forcing: 使用真实坐标（加少量噪声增加鲁棒性）
        placed_coords = lig_coords_true + torch.randn_like(lig_coords_true) * COORD_NOISE_STD

        # 构建已放置掩码（逐步放置，但 Teacher Forcing 下全部可见）
        mask = torch.ones(n_lig, device=prot_feats.device, dtype=torch.bool)

        # 上下文交互
        lig_h = self.encode_context(lig_h, prot_h, placed_coords, prot_coords, mask)

        # 逐原子预测位移
        per_atom_errors = []
        for step in range(1, len(atom_order)):
            child_idx = atom_order[step].item()
            parent_idx = parent_list[step].item()
            bond_len = bond_len_seq[step].item()

            displacement = self.predict_displacement(lig_h, child_idx, parent_idx, bond_len)
            pred_coord = placed_coords[parent_idx] + displacement

            # 与真实坐标的 L2 误差
            error = (pred_coord - lig_coords_true[child_idx]).pow(2).sum().sqrt()
            per_atom_errors.append(error)

        per_atom_loss = torch.stack(per_atom_errors)  # (N_l-1,)

        # 配位原子预测
        donor_pred = self.coord_donor_head(lig_h).squeeze(-1)  # (N_l,)

        return per_atom_loss, donor_pred

    def forward_inference(self, prot_feats, lig_feats, prot_coords,
                          atom_order, parent_list, bond_len_seq, pocket_center):
        """推理前向传播（自回归生成）。

        从第一个原子开始，逐步预测每个原子的坐标。
        每步用预测的坐标更新上下文（而非真实坐标）。

        返回:
          pred_coords : (N_l, 3)  预测的配体原子坐标
          donor_pred  : (N_l,)    配位原子预测概率
        """
        n_lig = lig_feats.size(0)

        # 嵌入
        prot_h = self.prot_embed(prot_feats)  # (N_p, hidden)
        lig_h = self.lig_embed(lig_feats)     # (N_l, hidden)

        # 初始化预测坐标
        pred_coords = torch.zeros(n_lig, 3, device=prot_feats.device)
        mask = torch.zeros(n_lig, device=prot_feats.device, dtype=torch.bool)

        # 第一个原子放在口袋中心
        first_atom = atom_order[0].item()
        pred_coords[first_atom] = pocket_center
        mask[first_atom] = True

        # 逐原子自回归生成
        for step in range(1, len(atom_order)):
            child_idx = atom_order[step].item()
            parent_idx = parent_list[step].item()
            bond_len = bond_len_seq[step].item()

            # 用当前已放置坐标编码上下文
            lig_h_ctx = self.encode_context(
                lig_h.clone(), prot_h, pred_coords.clone(), prot_coords, mask
            )

            # 预测位移
            displacement = self.predict_displacement(lig_h_ctx, child_idx, parent_idx, bond_len)
            pred_coords[child_idx] = pred_coords[parent_idx] + displacement
            mask[child_idx] = True

        # 配位原子预测（用最终上下文）
        lig_h_final = self.encode_context(lig_h, prot_h, pred_coords, prot_coords, mask)
        donor_pred = self.coord_donor_head(lig_h_final).squeeze(-1)  # (N_l,)

        return pred_coords, donor_pred

In [ ]:
# ============================================================
# 实例化模型并展示参数量
# ============================================================
model = ToyMetalloDock(atom_dim=ATOM_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(DEVICE)

# 统计各模块参数量
param_data = []
for name, param in model.named_parameters():
    param_data.append({
        '模块': name.split('.')[0],
        '参数名': name,
        '形状': str(tuple(param.shape)),
        '参数量': param.numel()
    })

param_df = pd.DataFrame(param_data)
total_params = sum(p.numel() for p in model.parameters())

# 按模块汇总
module_summary = param_df.groupby('模块')['参数量'].sum().reset_index()
module_summary.columns = ['模块', '参数量']
module_summary = pd.concat([
    module_summary,
    pd.DataFrame([{'模块': '总计', '参数量': total_params}])
], ignore_index=True)

print(f"模型总参数量: {total_params:,}")
display(module_summary)

## 5. 训练

训练采用 **Teacher Forcing** 策略：
- 训练时使用真实坐标（加少量高斯噪声）作为已放置原子的坐标
- 逐步预测每个原子的位移方向
- 损失函数 = 逐原子坐标误差均值 + 0.1 * 配位原子 BCE 损失
- 梯度裁剪 `max_norm=5.0` 防止梯度爆炸

In [ ]:
# ============================================================
# 训练循环
# ============================================================
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
bce_loss_fn = nn.BCELoss()

print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"开始训练 {N_EPOCHS} 轮...\n")

train_loss_history = []
val_loss_history = []
eval_epochs = []

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    train_losses = []

    for (prot_f, lig_f, prot_c, lig_c,
         atom_order, parent_list, bond_len_seq, donor_label) in train_loader:
        # 每个 batch 只有 1 个样本，去掉 batch 维度
        prot_f = prot_f.squeeze(0).to(DEVICE)           # (N_p, 10)
        lig_f = lig_f.squeeze(0).to(DEVICE)             # (N_l, 10)
        prot_c = prot_c.squeeze(0).to(DEVICE)           # (N_p, 3)
        lig_c = lig_c.squeeze(0).to(DEVICE)             # (N_l, 3)
        atom_order = atom_order.squeeze(0).to(DEVICE)   # (N_l,)
        parent_list = parent_list.squeeze(0).to(DEVICE) # (N_l,)
        bond_len_seq = bond_len_seq.squeeze(0).to(DEVICE)  # (N_l,)
        donor_label = donor_label.squeeze(0).to(DEVICE)    # (N_l,)

        if lig_f.size(0) < 2:
            continue

        optimizer.zero_grad()

        # Teacher Forcing 前向传播
        per_atom_loss, donor_pred = model.forward_train(
            prot_f, lig_f, prot_c, lig_c,
            atom_order, parent_list, bond_len_seq
        )

        # 损失: 逐原子坐标误差均值 + 配位原子 BCE
        coord_loss = per_atom_loss.mean()
        donor_loss = bce_loss_fn(donor_pred, donor_label)
        loss = coord_loss + 0.1 * donor_loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_losses.append(loss.item())

    # ---- 每 20 轮评估一次 ----
    if epoch % 20 == 0 or epoch == 1:
        model.eval()
        val_losses = []
        with torch.no_grad():
            for (prot_f, lig_f, prot_c, lig_c,
                 atom_order, parent_list, bond_len_seq, donor_label) in test_loader:
                prot_f = prot_f.squeeze(0).to(DEVICE)
                lig_f = lig_f.squeeze(0).to(DEVICE)
                prot_c = prot_c.squeeze(0).to(DEVICE)
                lig_c = lig_c.squeeze(0).to(DEVICE)
                atom_order = atom_order.squeeze(0).to(DEVICE)
                parent_list = parent_list.squeeze(0).to(DEVICE)
                bond_len_seq = bond_len_seq.squeeze(0).to(DEVICE)
                donor_label = donor_label.squeeze(0).to(DEVICE)

                if lig_f.size(0) < 2:
                    continue

                per_atom_loss, donor_pred = model.forward_train(
                    prot_f, lig_f, prot_c, lig_c,
                    atom_order, parent_list, bond_len_seq
                )
                coord_loss = per_atom_loss.mean()
                donor_loss = bce_loss_fn(donor_pred, donor_label)
                val_losses.append((coord_loss + 0.1 * donor_loss).item())

        avg_train = np.mean(train_losses) if train_losses else float('nan')
        avg_val = np.mean(val_losses) if val_losses else float('nan')
        train_loss_history.append(avg_train)
        val_loss_history.append(avg_val)
        eval_epochs.append(epoch)
        print(f"  Epoch {epoch:>4d}/{N_EPOCHS}  |  Train Loss: {avg_train:.4f}  |  Val Loss: {avg_val:.4f}")

## 6. 评估与可视化

使用 **自回归推理模式** 在测试集上评估：
- 从第一个原子（放在口袋中心）开始，逐步预测每个原子坐标
- 每步使用预测坐标（而非真实坐标）更新上下文
- 计算 RMSD（平移对齐后）评估对接精度

评估指标：
- **平均/中位数 RMSD**：整体对接精度
- **RMSD < 2A / < 5A 比例**：成功对接率
- **逐原子误差 vs BFS 步骤**：误差累积分析

In [ ]:
# ============================================================
# 测试集评估（自回归推理模式）
# ============================================================
print("在测试集上评估（自回归推理模式）...")

model.eval()

all_rmsds = []              # 每个复合物的 RMSD
all_per_atom_errors = []    # 所有原子的逐步误差（按 BFS 步骤编号聚合）
max_steps = 0

with torch.no_grad():
    for i, (prot_f, lig_f, prot_c, lig_c,
            atom_order, parent_list, bond_len_seq, donor_label) in enumerate(test_loader):
        prot_f = prot_f.squeeze(0).to(DEVICE)
        lig_f = lig_f.squeeze(0).to(DEVICE)
        prot_c = prot_c.squeeze(0).to(DEVICE)
        lig_c = lig_c.squeeze(0).to(DEVICE)
        atom_order = atom_order.squeeze(0).to(DEVICE)
        parent_list = parent_list.squeeze(0).to(DEVICE)
        bond_len_seq = bond_len_seq.squeeze(0).to(DEVICE)

        if lig_f.size(0) < 2:
            continue

        # 口袋中心作为第一个原子的初始位置
        pocket_center = prot_c.mean(dim=0)

        # 自回归推理
        pred_coords, _ = model.forward_inference(
            prot_f, lig_f, prot_c,
            atom_order, parent_list, bond_len_seq, pocket_center
        )

        # 计算整体 RMSD（平移对齐：将预测坐标的质心对齐到真实坐标的质心）
        pred_np = pred_coords.cpu().numpy()
        true_np = lig_c.cpu().numpy()

        pred_center = pred_np.mean(axis=0)
        true_center = true_np.mean(axis=0)
        pred_aligned = pred_np - pred_center + true_center

        rmsd = compute_rmsd(pred_aligned, true_np)
        all_rmsds.append(rmsd)

        # 逐原子误差（按 BFS 步骤编号）
        n_atoms = len(atom_order)
        max_steps = max(max_steps, n_atoms)
        per_atom_err = np.sqrt(np.sum((pred_aligned - true_np) ** 2, axis=1))
        # 按 BFS 顺序记录每步误差
        bfs_errors = []
        for step_idx in range(n_atoms):
            atom_idx = atom_order[step_idx].item()
            bfs_errors.append(per_atom_err[atom_idx])
        all_per_atom_errors.append(bfs_errors)

        if (i + 1) % 10 == 0:
            print(f"  评估进度: {i + 1}/{len(test_loader)}")

all_rmsds = np.array(all_rmsds)

# ---- 计算指标 ----
mean_rmsd = np.mean(all_rmsds)
median_rmsd = np.median(all_rmsds)
pct_under_2 = np.mean(all_rmsds < 2.0) * 100
pct_under_5 = np.mean(all_rmsds < 5.0) * 100

# 用 DataFrame 展示结果
results_df = pd.DataFrame({
    '指标': ['样本数', '平均 RMSD', '中位数 RMSD', 'RMSD < 2A', 'RMSD < 5A'],
    '值': [
        f'{len(all_rmsds)}',
        f'{mean_rmsd:.4f} A',
        f'{median_rmsd:.4f} A',
        f'{pct_under_2:.1f}%',
        f'{pct_under_5:.1f}%'
    ]
})
print("\n测试集结果:")
display(results_df)

In [ ]:
# ============================================================
# 可视化：两张子图
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 6))

# 子图 1: RMSD 直方图
ax1 = axes[0]
ax1.hist(all_rmsds, bins=20, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(mean_rmsd, color='red', linestyle='--', linewidth=1.5,
            label=f'Mean={mean_rmsd:.2f} A')
ax1.axvline(median_rmsd, color='orange', linestyle='--', linewidth=1.5,
            label=f'Median={median_rmsd:.2f} A')
ax1.set_xlabel('RMSD (A)', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('MetalloDock: Ligand Pose RMSD', fontsize=13)
ax1.legend(loc='upper right')

# 子图 2: 逐原子误差曲线（按 BFS 步骤编号取平均）
ax2 = axes[1]
# 将不同长度的序列对齐，按步骤编号取平均
step_errors = {}
for errors in all_per_atom_errors:
    for step, err in enumerate(errors):
        if step not in step_errors:
            step_errors[step] = []
        step_errors[step].append(err)

steps = sorted(step_errors.keys())
# 只绘制至少有 10 个样本的步骤
steps_filtered = [s for s in steps if len(step_errors[s]) >= 10]
mean_errors = [np.mean(step_errors[s]) for s in steps_filtered]

ax2.plot(steps_filtered, mean_errors, 'o-', color='steelblue', markersize=3, linewidth=1.5)
ax2.set_xlabel('BFS Step (atom generation order)', fontsize=12)
ax2.set_ylabel('Mean Per-Atom Error (A)', fontsize=12)
ax2.set_title('Per-Atom Error vs Generation Step', fontsize=13)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("可视化完成!")

## 总结

本教程实现了 MetalloDock 的核心思想——面向金属蛋白的自回归分子对接模型。

### 关键要点回顾

1. **BFS 拓扑排序**：通过广度优先搜索确定原子的生成顺序，从配位原子出发沿化学键拓扑展开，确保每个新原子都有已放置的父原子作为参考点。

2. **自回归生成**：每一步预测一个原子的 3D 坐标，新坐标 = 父原子坐标 + 归一化位移方向 * 键长。这种方式天然满足键长约束。

3. **Teacher Forcing vs 自回归推理**：训练时使用真实坐标（加噪声）避免误差累积，推理时使用预测坐标实现端到端生成。

4. **上下文交互层**：通过距离加权注意力机制，让已放置的配体原子感知蛋白口袋的空间环境，指导后续原子的放置方向。

5. **配位原子预测**：辅助任务预测哪些配体原子可能是金属配位原子（N/O/S），帮助模型理解金属蛋白特有的结合模式。

### 简化说明

本教程是 MetalloDock 的简化教学版本，与完整模型的主要差异：
- 使用简化的 10 维原子特征（完整版使用更丰富的原子/残基特征）
- 使用 MLP 替代 EGNN 进行上下文编码
- 配位原子标签采用启发式规则（完整版基于金属-配体距离）
- 在 CASF-2016 核心集上训练（完整版使用 PDBbind 全集）

### 输入与输出

- **输入**：蛋白口袋 PDB 文件 + 配体 MOL2/SDF 文件
- **输出**：预测的配体 3D 构象（原子坐标）+ 配位原子概率
- **评估指标**：RMSD（均方根偏差），越小表示预测构象越接近真实构象